# Barrido aleatorio por tamaño de conjunto de features

Para cada **tamaño** de conjunto (1 columna, 2, 3, …, hasta TAM_MAX), entrena N_POR_TAMANO
redes con columnas elegidas al azar de ese tamaño. Responde a: **¿cuántas columnas dan la
mejor red de media?** y **¿algún conjunto concreto despunta?**

Complementa al forward selection: este da el *panorama* (cuántas features convienen), el
forward selection da *el camino óptimo* (qué features concretas).

**Pensado para CPU** mientras esperas la GPU: modelos pequeños, rápido. Guarda cada modelo
en CSV al instante (si paras, no pierdes nada).

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, time, os, random
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler, StandardScaler

# ─────────── AJUSTAR ───────────
RUTA_DF        = "../data/csv/df_model_completo.csv"
TARGET_COL     = "eth_close_ret"
SEQ_LEN, HORIZON = 30, 3
TAM_MAX        = 25       # barrer de 1 a 25 columnas (>25 con 20 muestras es poco informativo)
N_POR_TAMANO   = 20       # combinaciones aleatorias por cada tamaño
MAX_EPOCHS     = 35
PACIENCIA_ES   = 5
SALIDA_CSV     = "barrido_por_tamano.csv"
SEMILLA        = 42
EXCLUIR = ["btc_open","btc_high","btc_low","btc_close","btc_volume",
           "eth_open","eth_high","eth_low","eth_close","eth_volume",
           "btc_mcap","eth_mcap","fear_greed","inflation","fed_rate"]
# ───────────────────────────────

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

df_model = pd.read_csv(RUTA_DF, parse_dates=["date"], index_col="date").sort_index()
candidatas = [c for c in df_model.columns if c not in EXCLUIR and c != TARGET_COL]
print(f"Columnas candidatas: {len(candidatas)}  |  filas: {len(df_model)}")
TAM_MAX = min(TAM_MAX, len(candidatas))

n=len(df_model); i_tr=int(n*0.70); i_va=int(n*0.85)
df_tr, df_va = df_model.iloc[:i_tr], df_model.iloc[i_tr:i_va]
sy = StandardScaler().fit(df_tr[TARGET_COL].values.reshape(-1,1))

In [ ]:
class LSTMReg(nn.Module):
    def __init__(self, nf, h=(48,24), horizon=HORIZON, dropout=0.35):
        super().__init__()
        capas=[]; ins=nf
        for hh in h: capas.append(nn.LSTM(ins,hh,batch_first=True)); ins=hh
        self.lstms=nn.ModuleList(capas)
        self.drops=nn.ModuleList([nn.Dropout(dropout) for _ in h])
        self.head=nn.Sequential(nn.Linear(h[-1],max(h[-1]//2,1)),nn.ReLU(),
                                nn.Dropout(dropout),nn.Linear(max(h[-1]//2,1),horizon))
    def forward(self,x):
        out=x
        for l,d in zip(self.lstms,self.drops): out,_=l(out); out=d(out)
        return self.head(out[:,-1,:])

class DS(Dataset):
    def __init__(s,X,y): s.X=torch.FloatTensor(X); s.y=torch.FloatTensor(y)
    def __len__(s): return len(s.X)
    def __getitem__(s,i): return s.X[i],s.y[i]

def _sec(X,y):
    Xs,ys=[],[]
    for i in range(SEQ_LEN,len(X)-HORIZON+1):
        Xs.append(X[i-SEQ_LEN:i]); ys.append(y[i:i+HORIZON])
    return np.array(Xs,np.float32),np.array(ys,np.float32)

def evaluar(cols_sel):
    torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)
    sx=RobustScaler().fit(df_tr[cols_sel].values)
    Xtr=sx.transform(df_tr[cols_sel].values); Xva=sx.transform(df_va[cols_sel].values)
    ytr=sy.transform(df_tr[TARGET_COL].values.reshape(-1,1)).ravel()
    yva=sy.transform(df_va[TARGET_COL].values.reshape(-1,1)).ravel()
    Xs_tr,ys_tr=_sec(Xtr,ytr); Xs_va,ys_va=_sec(Xva,yva)
    tl=DataLoader(DS(Xs_tr,ys_tr),batch_size=32,shuffle=True)
    vl=DataLoader(DS(Xs_va,ys_va),batch_size=64,shuffle=False)
    m=LSTMReg(len(cols_sel)).to(DEVICE)
    crit=nn.HuberLoss(); opt=torch.optim.Adam(m.parameters(),lr=8e-4,weight_decay=3e-4)
    mejor=float("inf"); sin=0
    for ep in range(MAX_EPOCHS):
        m.train()
        for xb,yb in tl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            loss=crit(m(xb),yb); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
        m.eval(); v=0
        with torch.no_grad():
            for xb,yb in vl:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE); v+=crit(m(xb),yb).item()*len(xb)
        v/=len(vl.dataset)
        if v<mejor-1e-5: mejor=v; sin=0
        else:
            sin+=1
            if sin>=PACIENCIA_ES: break
    m.eval(); P=[];T=[]
    with torch.no_grad():
        for xb,yb in vl:
            P.append(m(xb.to(DEVICE)).cpu().numpy()); T.append(yb.numpy())
    P=np.concatenate(P).ravel(); T=np.concatenate(T).ravel()
    return mejor, float(np.mean(np.sign(T)==np.sign(P)))

print("Funciones listas.")

In [ ]:
random.seed(SEMILLA)
if os.path.exists(SALIDA_CSV): os.remove(SALIDA_CSV)
with open(SALIDA_CSV,"w") as f: f.write("tamano,rep,val_loss,dir_acc,features\n")

t0=time.time()
for tam in range(1, TAM_MAX+1):
    for rep in range(N_POR_TAMANO):
        cols_sel=random.sample(candidatas, tam)
        try:
            vlo, da = evaluar(cols_sel)
            with open(SALIDA_CSV,"a") as f:
                f.write(f"{tam},{rep},{vlo:.5f},{da:.4f},\"{'|'.join(cols_sel)}\"\n")
        except Exception as e:
            print(f"  tam{tam} rep{rep} falló: {e}")
    print(f"Tamaño {tam:2d}/{TAM_MAX} hecho | {(time.time()-t0)/60:.1f} min", flush=True)

print(f"\nTerminado en {(time.time()-t0)/60:.1f} min")

In [ ]:
import matplotlib.pyplot as plt
r=pd.read_csv(SALIDA_CSV)
agg=r.groupby("tamano").agg(val_loss_medio=("val_loss","mean"),
                            val_loss_min=("val_loss","min"),
                            dir_acc_medio=("dir_acc","mean")).reset_index()

fig,ax1=plt.subplots(figsize=(11,5))
ax1.plot(agg["tamano"],agg["val_loss_medio"],"o-",color="#627EEA",label="val_loss medio")
ax1.fill_between(agg["tamano"],agg["val_loss_min"],agg["val_loss_medio"],alpha=0.15,color="#627EEA")
ax1.set_xlabel("nº de columnas"); ax1.set_ylabel("val_loss", color="#627EEA")
ax2=ax1.twinx()
ax2.plot(agg["tamano"],agg["dir_acc_medio"]*100,"s--",color="#E08A2B")
ax2.axhline(50,color="gray",ls=":",lw=0.6); ax2.set_ylabel("DirAcc % medio", color="#E08A2B")
ax1.set_title("Rendimiento por nº de columnas (media de 20 combinaciones aleatorias)")
plt.tight_layout(); plt.show()

print("Mejor val_loss medio en tamaño:", int(agg.loc[agg["val_loss_medio"].idxmin(),"tamano"]))
print("\nLos 5 MEJORES conjuntos individuales (de todos los tamaños):")
print(r.sort_values("val_loss").head(5)[["tamano","val_loss","dir_acc"]].to_string(index=False))
print("\n(El 'mejor individual' puede ser suerte entre cientos; fíate del val_loss MEDIO por tamaño.)")